# DataStax Astra DB

**Astra DB** is a **fully managed, serverless vector database** in the cloud, built on **Apache Cassandra** by DataStax (part of **IBM** since 2025). It is a popular choice for production RAG because:

- **Nothing to run yourself:** you create a database in the Astra web console and connect to it over HTTPS. Astra handles the servers, scaling, backups and replication.
- **Data is saved automatically:** every chunk you add is stored in the cloud straight away. There is no `save_local()` or `dump()`, and the data survives kernel restarts.
- **Shared by many users and apps:** any notebook, API or app with the endpoint and a token sees the same data.
- **Filtering inside the database:** metadata filters run as part of the search, like Chroma's `where`.
- **Extras for RAG:** it can compute the embeddings for you (**Vectorize**) and combine vector search with keyword search (**hybrid search**).
- **LangChain support:** the `langchain-astradb` package provides `AstraDBVectorStore`, which works like every other LangChain vector store.

The key difference from the previous notebooks: Chroma (with a local folder), FAISS and `InMemoryVectorStore` run **inside your Python process**. Astra DB runs **in the cloud**, and every add or search is a request over the network, authenticated with an **API endpoint** and a **token**.

> **In one sentence:** Astra DB is a cloud vector database you connect to with an endpoint and a token, so your vectors are stored, searched and shared without running any servers yourself.

This notebook uses `langchain-astradb` 1.0.x (`AstraDBVectorStore`), which talks to Astra DB's **Data API** through the `astrapy` 2.x client.

## Astra DB vs. the vector stores we've used so far

| | Astra DB | Chroma | FAISS | InMemoryVectorStore |
|---|---|---|---|---|
| **Where it runs** | In the cloud (AWS, Google Cloud or Azure), managed for you | Your process, or a Chroma server | Your process | Your process |
| **Setup** | Create a database online, then `langchain-astradb` + endpoint + token | `chromadb` + `langchain-chroma` | `faiss-cpu` | Nothing: part of `langchain-core` |
| **Saving** | Automatic, in the cloud | Automatic with `persist_directory` | Manual: `save_local()` | Manual: `dump()` |
| **Search** | Approximate (JVector graph index) | Approximate (HNSW) | Exact (Flat) or approximate | Exact |
| **Score from `similarity_search_with_score`** | Similarity from 0 to 1: **higher = better** | Distance: lower = better | Distance: lower = better | Cosine similarity: higher = better |
| **Metadata filtering** | Dict filters, **inside the database** | Dict filters (`where`), inside the database | After the search (`fetch_k`) | Python function, before the search |
| **Adding an existing id** | Overwrites it (upsert) | Overwrites it (upsert) | Raises `ValueError` | Overwrites it (upsert) |
| **Several users / apps** | Yes: anyone with the endpoint and a token | Yes, with a Chroma server | No | No |
| **Cost** | Free tier, then pay as you go | Free (open source), or Chroma Cloud | Free (open source) | Free |
| **Best for** | Production apps that need a managed, shared, always-on vector database | Local apps that need CRUD and filtering | Fast local search over large collections | Learning, tests, small prototypes |

The retrievers and LCEL chains from the previous notebooks work unchanged: only the line that creates the vector store changes.

## The building blocks: database, keyspace, collection, document

```
Astra organization (your account)
 │
 └── Database "rag-bootcamp"             serverless, in one cloud region
      │   API endpoint: https://<DB-ID>-<region>.apps.astra.datastax.com
      │
      └── Keyspace "default_keyspace"     groups collections (LangChain calls it `namespace`)
           │
           ├── Collection "rag_text_docs"      ← like a table: vector size and metric are fixed when it is created
           │     ├── document {"_id": "chunk_0", "content": "...", "$vector": [...], "metadata": {...}}
           │     ├── document {"_id": "chunk_1", ...}
           │     └── ...
           │
           └── Collection "..."
```

`AstraDBVectorStore` stores each chunk as one JSON document:

| Field | What it holds | Example |
|---|---|---|
| `_id` | The chunk id | `"chunk_0"` |
| `content` | The chunk text | `"AI Agents: An Introduction ..."` |
| `$vector` | The embedding | 1536 numbers for `text-embedding-3-small` |
| `metadata` | The chunk's metadata | `{"source": "/.../RAG-Bootcamp/data/raw/text/ai_agents.txt", "topic": "agents"}` |

- **Database:** created in the Astra console, in the cloud provider and region you choose. Each database has its own **API endpoint**.
- **Keyspace:** a namespace for collections. New databases come with `default_keyspace`, which LangChain uses when you don't pass one.
- **Collection:** a set of documents with the **same vector size** and the **same similarity metric**, both fixed when the collection is created. A database holds about 10 collections.
- **Document:** one chunk. The `metadata` dict is what you filter on.

## Connecting: API endpoint and token

You need two values from the Astra console at [astra.datastax.com](https://astra.datastax.com):

1. **Create a database:** choose *Serverless (vector)*, a name, a cloud provider and a region. It takes a few minutes to become *Active*.
2. **API endpoint:** copy it from the database's overview page, e.g. `https://01234567-89ab-cdef-0123-456789abcdef-us-east1.apps.astra.datastax.com`.
3. **Application token:** generate one on the same page. It starts with `AstraCS:`. Give it a role that can create collections, such as *Database Administrator*.

Put both in your `.env` file, next to `OPENAI_API_KEY`:

```
ASTRA_DB_API_ENDPOINT=https://01234567-89ab-cdef-0123-456789abcdef-us-east1.apps.astra.datastax.com
ASTRA_DB_APPLICATION_TOKEN=AstraCS:...
ASTRA_DB_KEYSPACE=default_keyspace      # optional
```

`AstraDBVectorStore` reads these three environment variables when you don't pass `api_endpoint`, `token` or `namespace`. Install the package with `uv add langchain-astradb` (or `pip install langchain-astradb`).

> **The token is a password.** Anyone who has it can read and delete your data. Keep it in `.env` (which git ignores) and never paste it into a notebook cell.

## Two ways to embed: client-side or Vectorize

```
Client-side: embedding=OpenAIEmbeddings(...)      Vectorize: collection_vector_service_options=...

chunk text                                        chunk text
   │  embedded in your notebook                      │  sent as plain text
   ▼                                                 ▼
vector ──► Astra DB stores it in "$vector"        Astra DB calls the embedding provider itself
                                                  and stores the result in "$vector"
```

| | Client-side | Vectorize |
|---|---|---|
| **Who calls the embedding model** | Your Python code | Astra DB |
| **Set up with** | `embedding=OpenAIEmbeddings(...)` | `collection_vector_service_options=VectorServiceOptions(...)` |
| **Where the OpenAI key lives** | Your `.env` | In Astra, as an embedding provider integration |
| **What you send to Astra** | Text and vectors | Text only |

```python
from astrapy.info import VectorServiceOptions

vectorstore = AstraDBVectorStore(
    collection_name="rag_text_docs_vectorize",
    collection_vector_service_options=VectorServiceOptions(
        provider="openai",
        model_name="text-embedding-3-small",
        authentication={"providerKey": "OPENAI_API_KEY"},   # the name of the key you saved in Astra
    ),
)
```

You pass **either** `embedding` **or** `collection_vector_service_options`, never both. This notebook uses **client-side** embeddings, with the same `OpenAIEmbeddings` model as the previous notebooks.

## Similarity metrics and scores

The **metric** is chosen when the collection is created and can't be changed afterwards:

| `metric` | Notes |
|---|---|
| `"cosine"` | **The default.** Works for any embeddings |
| `"dot_product"` | Faster than cosine, but only correct for **normalised** vectors (OpenAI embeddings are normalised) |
| `"euclidean"` | Based on the straight-line distance between the vectors |

Astra uses a **JVector** graph index (inspired by DiskANN) for **approximate** nearest-neighbour search, so it stays fast with millions of vectors.

**Scores:** `similarity_search_with_score` returns a **similarity from 0 to 1: higher = better**. That is the same direction as `InMemoryVectorStore`, but on a different scale. With `cosine`, Astra rescales the cosine similarity:

$$\text{Astra score} = \frac{1 + \text{cosine similarity}}{2}$$

| Cosine similarity | Astra score | Meaning |
|---|---|---|
| `1.0` | `1.0` | Same meaning |
| `0.68` | `0.84` | A good match, like the top chunk for *"What is reinforcement learning?"* in the previous notebooks |
| `0` | **`0.5`** | Unrelated |
| `-1.0` | `0` | Opposite |

So **unrelated chunks score about 0.5, not 0**. The ranking is the same as in the other stores, only the numbers differ.

**Relevance scores work:** `similarity_search_with_relevance_scores` and `as_retriever(search_type="similarity_score_threshold")` use this score as it is. Because 0.5 already means "unrelated", choose a threshold well above 0.5, for example:

```python
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.8},
)
```

## Working with AstraDBVectorStore in LangChain

| Method | What it does |
|---|---|
| `AstraDBVectorStore(collection_name, embedding, ...)` | Connect to a collection, creating it if it doesn't exist |
| `AstraDBVectorStore.from_documents(docs, embedding, ids=..., collection_name=...)` | Create the store and add the chunks in one step |
| `vectorstore.add_documents(docs, ids=...)` | Embed and add chunks. An id that already exists is **overwritten** |
| `vectorstore.similarity_search(query, k=4, filter=...)` | Return the `k` most similar `Document`s (default `k=4`) |
| `vectorstore.similarity_search_with_score(query, k=4, filter=...)` | Return `(Document, score)` pairs, with scores from 0 to 1 |
| `vectorstore.max_marginal_relevance_search(query, k, fetch_k, filter=...)` | Return relevant **and** varied chunks (MMR) |
| `vectorstore.metadata_search(filter, n=5)` | Find chunks by metadata only, without a vector search |
| `vectorstore.get_by_document_id(id)` | Fetch one chunk by id |
| `vectorstore.update_metadata({id: {...}})` | Add or change metadata keys of existing chunks |
| `vectorstore.delete(ids=[...])` | Remove chunks by id |
| `vectorstore.delete_by_metadata_filter(filter)` | Remove every chunk that matches a filter |
| `vectorstore.clear()` | Remove all chunks but keep the collection |
| `vectorstore.delete_collection()` | Delete the whole collection and its data |
| `vectorstore.as_retriever(search_kwargs={"k": 3})` | Use it in a RAG chain, like the other retrievers |

Every method also has an async version: `aadd_documents`, `asimilarity_search`, `adelete`, ...

### Connect, add and reconnect

```python
import os
from langchain_astradb import AstraDBVectorStore

# Connect to the collection (it is created the first time)
vectorstore = AstraDBVectorStore(
    collection_name="rag_text_docs",
    embedding=embeddings,
    api_endpoint=os.environ["ASTRA_DB_API_ENDPOINT"],
    token=os.environ["ASTRA_DB_APPLICATION_TOKEN"],
)

# Add the chunks: they are stored in the cloud straight away
vectorstore.add_documents(chunks, ids=[f"chunk_{i}" for i in range(len(chunks))])

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
```

Later, in another notebook or app, run the same `AstraDBVectorStore(...)` call: it connects to the existing collection, and the chunks are already there. There is nothing to save or load.

### Things to watch out for

- **Settings are fixed per collection.** The vector size, metric, Vectorize settings and indexing are set when the collection is created. Connecting to an existing collection with different settings fails with an `EXISTING_COLLECTION_DIFFERENT_SETTINGS` error. Use a new collection name, or pass `autodetect_collection=True` to read the settings from the existing collection.
- **Fixed ids prevent duplicates.** `add_documents` with ids that already exist replaces those chunks. Without ids, every run creates new random ids, so running the cell twice **duplicates** your chunks, and they stay in the cloud.
- **Every call goes over the network.** Each search embeds the question and then sends a request to Astra, so it is slower than an in-process FAISS search. LangChain sends inserts in concurrent batches to keep `add_documents` fast.
- **Same embedding model.** As with every vector store, query with the model that built the collection. Astra also rejects vectors of the wrong size.
- **Deleting is permanent.** `clear()`, `delete_collection()` and `pre_delete_collection=True` remove the data in the cloud, for every user of the collection.

## Metadata filtering

Filters are **dicts** with MongoDB-style operators, and they run **inside the database** as part of the search. You get `k` results whenever at least `k` chunks match, and there is no `fetch_k` to tune:

```python
# One topic
results = vectorstore.similarity_search("What is reinforcement learning?", k=3, filter={"topic": "ml"})

# ml or dl
results = vectorstore.similarity_search(
    "What is reinforcement learning?",
    k=3,
    filter={"topic": {"$in": ["ml", "dl"]}},
)

# The same filter in a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"topic": "dl"}})
```

| Operator | Meaning | Example |
|---|---|---|
| `$eq`, `$ne` | equal / not equal | `{"topic": "ml"}` (short for `$eq`) |
| `$gt`, `$gte`, `$lt`, `$lte` | greater / less than | `{"page": {"$gte": 3}}` |
| `$in`, `$nin` | in / not in a list | `{"topic": {"$in": ["ml", "dl"]}}` |
| `$exists` | the field is present | `{"topic": {"$exists": True}}` |
| `$all`, `$size` | an array contains all the values / has this many items | `{"tags": {"$all": ["rag", "faiss"]}}` |
| `$and`, `$or`, `$not` | combine conditions | `{"$or": [{"topic": "ml"}, {"topic": "dl"}]}` |

LangChain adds the `metadata.` prefix for you: `{"topic": "ml"}` is sent to Astra as `{"metadata.topic": "ml"}`.

How this compares with the other stores:

| | Where the filter runs | Can you get fewer than `k` results? |
|---|---|---|
| FAISS | In Python, **after** the search, on the `fetch_k` nearest chunks | Yes, when few of those chunks match |
| InMemoryVectorStore | In Python, **before** the search | Only when fewer than `k` chunks match |
| Astra DB | **Inside the database**, during the search | Only when fewer than `k` chunks match |

## Hybrid search

Vector search finds chunks with a **similar meaning**, but it can miss exact words such as product codes, error messages or names. Astra DB can also run a **hybrid search**:

```
question ─┬─► vector search    (meaning)   ─┐
          │                                 ├─► reranker model ─► top k
          └─► lexical search   (keywords)  ─┘
```

To use it, create the collection with a keyword analyzer (`collection_lexical`) and a reranker (`collection_rerank`). After that, `similarity_search` runs hybrid searches automatically, and you can pass a separate keyword query with `lexical_query="..."`. We'll look at hybrid search in detail later in the course.

## Strengths and limits

**Strengths**

- Fully managed and serverless: no servers, scaling or backups to handle
- Data is saved automatically and shared by every app and user with access
- Scales to millions of vectors with an approximate (JVector) index
- Metadata filters with rich operators, run inside the database
- Upsert, delete by id or by filter, and metadata updates
- Vectorize (server-side embeddings) and hybrid search built in
- Runs on AWS, Google Cloud and Azure, with a free tier to start

**Limits to keep in mind**

- **Needs an account and the internet.** Every call is a network request, and your app depends on a cloud service.
- **Slower per query than in-process stores.** A network round trip costs more than a FAISS or in-memory search, which matters for small local projects.
- **Costs grow with use.** Beyond the free tier, you pay for reads, writes and storage.
- **Fixed collection settings.** Changing the vector size, metric or embedding model means creating a new collection and adding the data again.
- **Limits per database.** About 10 collections per database, and a vector search returns at most 1,000 documents.
- **Secrets to manage.** The application token gives access to your data, so it must stay out of your code and out of git.

## Key takeaways

- **Astra DB** is a serverless **cloud** vector database built on Apache Cassandra, from DataStax (now part of IBM). `AstraDBVectorStore` from `langchain-astradb` plugs it into LangChain.
- You connect with an **API endpoint** and an **application token**, kept in `.env` as `ASTRA_DB_API_ENDPOINT` and `ASTRA_DB_APPLICATION_TOKEN`.
- Data lives in **collections** of JSON documents (`_id`, `content`, `$vector`, `metadata`). The vector size and the **metric** are fixed when a collection is created.
- **Saving is automatic** and the data is **shared**: reconnect with the same collection name and everything is there.
- Scores go from **0 to 1, higher = better**. With `cosine`, unrelated chunks score about **0.5**. Relevance-score thresholds work.
- `add_documents` with existing ids **overwrites** them, so fixed ids prevent duplicates.
- **Metadata filters** are dicts with MongoDB-style operators, run **inside the database**.
- **Vectorize** can embed on the server, and **hybrid search** combines vector and keyword search.
- `as_retriever()` plugs Astra DB into the same LCEL and conversational RAG chains we built with Chroma, FAISS and `InMemoryVectorStore`.

**Next:** we'll connect to Astra DB, add our text chunks to a collection, search them with and without filters, and use the collection in a RAG chain.

## Hands-on: a RAG pipeline on Astra DB

Every step is a small helper function, so the notebook reads top to bottom and the helpers are easy to reuse:

| Step | Helpers |
|---|---|
| 1. Connection | `get_astra_vectorstore()`, `show_collections()` |
| 2. Data ingestion | `load_file()`, `load_all_files()`, `split_documents()`, `make_chunk_ids()`, `ingest_chunks()`, `run_ingestion()` |
| 3. Retrieval pipeline | `source_label()`, `get_retriever()`, `search_with_scores()` |
| 4. RAG chain | `format_docs()`, `build_rag_chain()` |
| 5. RAG pipeline | `create_rag_pipeline()`, `ask()` |
| 6. Conversational RAG | `build_conversational_rag_chain()`, `create_conversational_rag_pipeline()`, `chat()`, `show_chat_history()` |

`load_all_files()` loads **every supported file** in `data/raw` and its sub-folders, not only text:

| Extension | Loader | One Document per |
|---|---|---|
| `.txt` | `TextLoader` | file |
| `.pdf` | `PyPDFLoader` | page |
| `.docx` | `Docx2txtLoader` | file |
| `.csv` | `CSVLoader` | row |
| `.xlsx` | `load_excel()` (pandas) | row of each sheet |
| `.json` | `load_json()`: flattened to `departments.data_science.head: Sarah Williams` lines | file |
| `.jsonl` | `JSONLoader(json_lines=True)` | line |
| `.db` (SQLite) | `load_sqlite()` (`SQLDatabaseLoader`) | row of each table |

Every chunk gets the same small set of metadata, so you can filter on it: `source`, `file_name`, `file_type`, plus `page`, `row`, `sheet` or `table` when they apply.

You need `ASTRA_DB_API_ENDPOINT`, `ASTRA_DB_APPLICATION_TOKEN` and `OPENAI_API_KEY` in `.env`.

In [1]:
## Setup: shared project paths from src.config, which also loads the API keys from .env
import os
from pathlib import Path
from dotenv import load_dotenv

from src.config import PROJECT_ROOT, RAW_DATA_DIR

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [2]:
## Load libraries
import warnings
warnings.filterwarnings("ignore")

import json
from collections import defaultdict

import pandas as pd
from astrapy import DataAPIClient
from astrapy.exceptions import TooManyDocumentsToCountException

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableBranch
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_astradb import AstraDBVectorStore
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    CSVLoader,
    JSONLoader,
    SQLDatabaseLoader,
)
from langchain_community.utilities import SQLDatabase

In [3]:
## Settings used by the helpers
DATA_DIR = RAW_DATA_DIR
COLLECTION_NAME = "rag_bootcamp_docs"   # the Astra DB collection used by this notebook

EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-5"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 4

## 1. Connection

In [4]:
## Helpers: connect to Astra DB and list the collections
def get_astra_vectorstore(embedding, collection_name=COLLECTION_NAME):
    """Connect to Astra DB with the keys from .env and open the collection (it is created if it doesn't exist)."""
    load_dotenv(PROJECT_ROOT / ".env")   # picks up keys added to .env after the kernel started
    missing = [name for name in ("ASTRA_DB_API_ENDPOINT", "ASTRA_DB_APPLICATION_TOKEN") if not os.getenv(name)]
    if missing:
        raise ValueError(f"Add {', '.join(missing)} to the .env file in the project root and save the file")

    vectorstore = AstraDBVectorStore(
        collection_name=collection_name,
        embedding=embedding,
        api_endpoint=os.getenv("ASTRA_DB_API_ENDPOINT"),
        token=os.getenv("ASTRA_DB_APPLICATION_TOKEN"),
        # keyspace: ASTRA_DB_KEYSPACE from .env, or the database's default keyspace
    )
    print(f"Connected to Astra DB | collection: {collection_name}")
    return vectorstore


def show_collections():
    """Print every collection in the keyspace with its number of documents."""
    database = DataAPIClient().get_database(
        os.getenv("ASTRA_DB_API_ENDPOINT"),
        token=os.getenv("ASTRA_DB_APPLICATION_TOKEN"),
        keyspace=os.getenv("ASTRA_DB_KEYSPACE"),
    )
    names = database.list_collection_names()
    print(f"{len(names)} collection(s) in keyspace '{database.keyspace}':")
    for name in names:
        try:
            count = database.get_collection(name).count_documents({}, upper_bound=1000)
        except TooManyDocumentsToCountException:
            count = "more than 1000"
        print(f"  - {name}: {count} documents")

## 2. Data ingestion

In [5]:
## Helpers: load every supported file type, keep simple metadata, split, and upsert into Astra DB
def load_excel(path):
    """One Document per row of every sheet, written as "column: value" lines (like CSVLoader)."""
    documents = []
    for sheet_name, df in pd.read_excel(path, sheet_name=None).items():
        for row_number, row in df.iterrows():
            text = "\n".join(f"{column}: {value}" for column, value in row.items())
            documents.append(Document(page_content=text, metadata={"sheet": sheet_name, "row": int(row_number)}))
    return documents


def flatten_json(value, prefix=""):
    """Yield ("path.to.key", value) pairs for every value in a JSON object. Lists of plain values are joined."""
    if isinstance(value, dict):
        for key, item in value.items():
            yield from flatten_json(item, f"{prefix}.{key}" if prefix else key)
    elif isinstance(value, list) and all(not isinstance(item, (dict, list)) for item in value):
        yield prefix, ", ".join(map(str, value))
    elif isinstance(value, list):
        for index, item in enumerate(value):
            yield from flatten_json(item, f"{prefix}[{index}]")
    else:
        yield prefix, value


def load_json(path):
    """The whole JSON file as "departments.data_science.head: Sarah Williams" lines, which embed much better than raw JSON."""
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return [Document(page_content="\n".join(f"{key}: {value}" for key, value in flatten_json(data)))]


def load_sqlite(path):
    """One Document per row of every table, written as "column: value" lines."""
    database = SQLDatabase.from_uri(f"sqlite:///{path}")
    documents = []
    for table in database.get_usable_table_names():
        for doc in SQLDatabaseLoader(query=f"SELECT * FROM {table}", db=database).load():
            doc.metadata["table"] = table
            documents.append(doc)
    return documents


# File extension -> function that loads the file as a list of Documents
LOADERS = {
    ".txt":   lambda path: TextLoader(path, encoding="utf-8").load(),
    ".pdf":   lambda path: PyPDFLoader(path).load(),                    # one Document per page
    ".docx":  lambda path: Docx2txtLoader(path).load(),
    ".csv":   lambda path: CSVLoader(path, encoding="utf-8").load(),    # one Document per row
    ".xlsx":  load_excel,                                               # one Document per row of each sheet
    ".json":  load_json,                                                # nested keys flattened to "a.b.c: value" lines
    ".jsonl": lambda path: JSONLoader(path, jq_schema=".", text_content=False, json_lines=True).load(),   # one per line
    ".db":    load_sqlite,                                              # one Document per row of each table
}

# Loader metadata worth keeping. Everything else (PDF producer, creation date, ...) is dropped
KEEP_METADATA = ("page", "row", "sheet", "table")


def load_file(path):
    """Load one file with the loader for its extension, and give every Document the same simple metadata."""
    path = Path(path)
    documents = LOADERS[path.suffix.lower()](str(path))
    for doc in documents:
        doc.metadata = {
            "source": str(path),
            "file_name": path.name,
            "file_type": path.suffix.lower().lstrip("."),
            **{key: doc.metadata[key] for key in KEEP_METADATA if key in doc.metadata},
        }
    return documents


def load_all_files(data_dir=DATA_DIR):
    """Load every supported file in data_dir and its sub-folders. Other files (.DS_Store, ...) are skipped."""
    documents = []
    for path in sorted(Path(data_dir).rglob("*")):
        if path.suffix.lower() in LOADERS:
            file_docs = load_file(path)
            documents.extend(file_docs)
            print(f"  {path.relative_to(data_dir)}: {len(file_docs)} documents")
    print(f"Loaded {len(documents)} documents from {data_dir}")
    return documents


def split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Split long documents into chunks. Short ones (a CSV row, a JSONL line) stay as one chunk."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],   # try paragraphs first, then lines, then words, then characters
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(chunks)} chunks")
    return chunks


def make_chunk_ids(chunks):
    """Stable ids per file (proposal.docx_chunk_0, ...), so re-running the ingestion overwrites instead of duplicating."""
    position = defaultdict(int)
    ids = []
    for chunk in chunks:
        file_name = chunk.metadata["file_name"]
        ids.append(f"{file_name}_chunk_{position[file_name]}")
        position[file_name] += 1
    return ids


def ingest_chunks(vectorstore, chunks, ids, batch_size=100):
    """Embed the chunks and upsert them into Astra DB, batch_size chunks per call."""
    for start in range(0, len(chunks), batch_size):
        end = min(start + batch_size, len(chunks))
        vectorstore.add_documents(documents=chunks[start:end], ids=ids[start:end])
        print(f"  upserted chunks {start + 1} to {end} of {len(chunks)}")


def run_ingestion(vectorstore, data_dir=DATA_DIR):
    """Data ingestion pipeline: load every file -> split -> stable ids -> embed and upsert. Returns the chunks."""
    chunks = split_documents(load_all_files(data_dir))
    ingest_chunks(vectorstore, chunks, make_chunk_ids(chunks))
    return chunks

## 3. Retrieval pipeline

Astra DB scores go from **0 to 1: higher = more similar** (with the default `cosine` metric, unrelated chunks score about 0.5). Metadata filters are dicts that run inside the database, for example `{"file_type": "pdf"}` or `{"file_type": {"$in": ["csv", "xlsx"]}}`.

In [6]:
## Helpers: a readable source label, a retriever with an optional filter, and a scored search
def source_label(doc):
    """File name plus page, sheet, table or row when there is one, e.g. "1706.03762v7.pdf p.3"."""
    meta = doc.metadata
    label = meta.get("file_name", "unknown")
    if "page" in meta:
        label += f" p.{meta['page'] + 1}"   # PyPDFLoader counts pages from 0
    if "sheet" in meta:
        label += f" sheet {meta['sheet']}"
    if "table" in meta:
        label += f" table {meta['table']}"
    if "row" in meta:
        label += f" row {meta['row']}"
    return label


def get_retriever(vectorstore, k=TOP_K, filter=None):
    """A retriever for the k most similar chunks, optionally only the ones that match a metadata filter."""
    search_kwargs = {"k": k}
    if filter:
        search_kwargs["filter"] = filter
    return vectorstore.as_retriever(search_kwargs=search_kwargs)


def search_with_scores(vectorstore, query, k=TOP_K, filter=None):
    """Print the k closest chunks with their score (0 to 1, higher = more similar)."""
    results = vectorstore.similarity_search_with_score(query, k=k, filter=filter)

    print(f"Query : {query}")
    print(f"Filter: {filter}\n")
    for i, (doc, score) in enumerate(results, start=1):
        print(f"--- Result {i} | score: {score:.4f} | {source_label(doc)} ---")
        print(doc.page_content[:200] + "...\n")
    return results

## 4. RAG chain

In [7]:
## Helpers: turn the chunks into prompt context and build the LCEL RAG chain
rag_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Mention the source(s) you used.

Context:
{context}

Question: {question}

Answer:""")


def format_docs(docs):
    """Join the retrieved chunks into one string, each labelled with its source."""
    return "\n\n".join(f"[Source: {source_label(doc)}]\n{doc.page_content}" for doc in docs)


def build_rag_chain(retriever, llm, prompt=rag_prompt):
    """LCEL chain: question -> {"docs": [Document, ...], "question": "...", "answer": "..."}."""
    answer_chain = (
        RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))   # Documents -> context string
        | prompt
        | llm
        | StrOutputParser()
    )
    return (
        {"docs": retriever, "question": RunnablePassthrough()}
        | RunnablePassthrough.assign(answer=answer_chain)   # adds "answer", keeps "docs" and "question"
    )

## 5. RAG pipeline

In [8]:
## Helpers: the full RAG pipeline (retrieve + generate) in one call, and an async ask()
def create_rag_pipeline(vectorstore, k=TOP_K, filter=None, model=LLM_MODEL):
    """Retriever + LLM + RAG chain, ready to use with ask()."""
    retriever = get_retriever(vectorstore, k=k, filter=filter)
    llm = ChatOpenAI(model=model, temperature=0)
    return build_rag_chain(retriever, llm)


async def ask(rag_pipeline, question):
    """Run the pipeline with ainvoke, then print the answer and the sources of the chunks it used."""
    result = await rag_pipeline.ainvoke(question)

    print(f"Question: {question}\n")
    print(f"Answer:\n{result['answer']}\n")
    print("Sources:")
    for doc in result["docs"]:
        print(f"  - {source_label(doc)}")
    return result

## 6. Conversational RAG (chat memory)

`ask()` forgets each question as soon as it answers it, so a follow-up such as *"What is its budget?"* fails. The conversational pipeline keeps the **chat history** and uses it twice:

```
{"question": "What is its budget?", "chat_history": [...]}
        │
        ▼
 1. condense_question    no history   → use the question as it is
        │                with history → the LLM rewrites it: "What is the budget of the RAG Implementation project?"
        ▼
 2. retriever            Astra DB searches with the standalone question → docs
        │
        ▼
 3. answer               qa_prompt (context + chat_history + question) | llm | StrOutputParser()
```

`chat()` saves every question and answer to a `chat_history` list, and sends only the last `max_turns` turns, so long conversations don't make the prompt huge.

In [9]:
## Helpers: conversational RAG chain with chat memory
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, which might refer to the "
     "chat history, rewrite it as a standalone question that can be understood without "
     "the chat history. Do NOT answer the question: only rewrite it if needed, "
     "otherwise return it as is."),
    MessagesPlaceholder("chat_history"),   # the past HumanMessage / AIMessage objects go here
    ("human", "{question}"),
])

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer using only the context below. "
     "If the answer is not in the context, say you don't know. Mention the source(s) you used.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])


def build_conversational_rag_chain(retriever, llm):
    """LCEL chain: {"question", "chat_history"} -> adds "standalone_question", "docs" and "answer"."""
    condense_question = RunnableBranch(
        (lambda x: not x["chat_history"], lambda x: x["question"]),   # no history: search with the question as it is
        contextualize_q_prompt | llm | StrOutputParser(),              # with history: let the LLM rewrite it first
    )
    answer_chain = (
        RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))   # Documents -> context string
        | qa_prompt
        | llm
        | StrOutputParser()
    )
    return (
        RunnablePassthrough.assign(standalone_question=condense_question)
        | RunnablePassthrough.assign(docs=(lambda x: x["standalone_question"]) | retriever)
        | RunnablePassthrough.assign(answer=answer_chain)
    )


def create_conversational_rag_pipeline(vectorstore, k=TOP_K, filter=None, model=LLM_MODEL):
    """Retriever + LLM + conversational RAG chain, ready to use with chat()."""
    retriever = get_retriever(vectorstore, k=k, filter=filter)
    llm = ChatOpenAI(model=model, temperature=0)
    return build_conversational_rag_chain(retriever, llm)


async def chat(conversational_pipeline, question, chat_history, max_turns=5):
    """Answer with the last max_turns turns as memory, save this turn to chat_history, and print what was searched."""
    recent_history = chat_history[-2 * max_turns:]   # one turn = a HumanMessage + an AIMessage
    result = await conversational_pipeline.ainvoke({"question": question, "chat_history": recent_history})
    chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])

    print(f"Question     : {question}")
    print(f"Searched with: {result['standalone_question']}\n")
    print(f"Answer:\n{result['answer']}\n")
    print("Sources:")
    for doc in result["docs"]:
        print(f"  - {source_label(doc)}")
    print()
    return result


def show_chat_history(chat_history):
    """Print the conversation so far: one line per message."""
    print(f"{len(chat_history)} messages in the history\n")
    for message in chat_history:
        print(f"{message.type:>5}: {message.content[:100].replace('\n', ' ')}...")

## 7. Run it

In [10]:
## Connect to Astra DB
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = get_astra_vectorstore(embeddings)

show_collections()

Connected to Astra DB | collection: rag_bootcamp_docs
1 collection(s) in keyspace 'default_keyspace':
  - rag_bootcamp_docs: 427 documents


In [11]:
## Data ingestion: load every file in data/raw, split, embed and upsert (only needed once, or when the files change)
chunks = run_ingestion(vectorstore)

show_collections()

  databases/company.db: 41 documents
  json_files/company_data.json: 1 documents
  json_files/events.jsonl: 15 documents
  pdf/1706.03762v7.pdf: 15 documents
  pdf/2403.08299v1.pdf: 10 documents
  pdf/2609.19135v1.pdf: 17 documents
  structured_files/inventory.xlsx: 12 documents
  structured_files/products.csv: 10 documents
  text/ai_agents.txt: 1 documents
  text/deep_learning.txt: 1 documents
  text/machine_learning_algorithms.txt: 1 documents
  text/reinforcement_learning.txt: 1 documents
  word_files/proposal.docx: 1 documents
Loaded 126 documents from /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw
Split 126 documents into 427 chunks
  upserted chunks 1 to 100 of 427
  upserted chunks 101 to 200 of 427
  upserted chunks 201 to 300 of 427
  upserted chunks 301 to 400 of 427
  upserted chunks 401 to 427 of 427
1 collection(s) in keyspace 'default_keyspace':
  - rag_bootcamp_docs: 427 documents


In [12]:
## Retrieval: check what comes back before adding the LLM
results = search_with_scores(vectorstore, "What is multi-head attention?")

Query : What is multi-head attention?
Filter: None

--- Result 1 | score: 0.8352 | 1706.03762v7.pdf p.5 ---
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from diff...

--- Result 2 | score: 0.8218 | 1706.03762v7.pdf p.4 ---
Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, ...

--- Result 3 | score: 0.8154 | 1706.03762v7.pdf p.2 ---
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
descr...

--- Result 4 | score: 0.7892 | 1706.03762v7.pdf p.4 ---
extremely small gradients 4. To counteract this effect, we scale the dot products by 1√dk
.
3.2.2 Multi-Head 

In [13]:
## Retrieval with a metadata filter: only chunks from CSV and Excel files
results = search_with_scores(
    vectorstore,
    "How many laptops are in stock and what do they cost?",
    filter={"file_type": {"$in": ["csv", "xlsx"]}},
)

Query : How many laptops are in stock and what do they cost?
Filter: {'file_type': {'$in': ['csv', 'xlsx']}}

--- Result 1 | score: 0.7838 | products.csv row 0 ---
Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD...

--- Result 2 | score: 0.7837 | inventory.xlsx sheet Products row 0 ---
Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD...

--- Result 3 | score: 0.6931 | inventory.xlsx sheet Products row 8 ---
Product: External SSD
Category: Electronics
Price: 129.99
Stock: 90
Description: 1TB portable external SSD with USB-C connectivity...

--- Result 4 | score: 0.6931 | products.csv row 8 ---
Product: External SSD
Category: Electronics
Price: 129.99
Stock: 90
Description: 1TB portable external SSD with USB-C connectivity...



In [14]:
## RAG pipeline over every file type
rag_pipeline = create_rag_pipeline(vectorstore)

result = await ask(rag_pipeline, "What is multi-head attention?")

Question: What is multi-head attention?

Answer:
Multi-head attention is an attention mechanism where, instead of applying a single attention function with d_model-dimensional queries, keys, and values, the model linearly projects Q, K, and V h times with different learned projections to lower dimensions (d_k, d_k, d_v), applies attention in parallel on each projected set (forming h “heads”), concatenates the resulting outputs, and applies a final linear projection. This lets the model jointly attend to information from different representation subspaces at different positions.

Formally:
- MultiHead(Q, K, V) = Concat(head_1, ..., head_h) W^O
- head_i = Attention(Q W_i^Q, K W_i^K, V W_i^V)
- W_i^Q ∈ R^{d_model×d_k}, W_i^K ∈ R^{d_model×d_k}, W_i^V ∈ R^{d_model×d_v}

Sources:
- 1706.03762v7.pdf p.4
- 1706.03762v7.pdf p.5
- 1706.03762v7.pdf p.2

Sources:
  - 1706.03762v7.pdf p.5
  - 1706.03762v7.pdf p.4
  - 1706.03762v7.pdf p.2
  - 1706.03762v7.pdf p.4


In [15]:
result = await ask(rag_pipeline, "Who is the head of the data science department and what is its budget?")

Question: Who is the head of the data science department and what is its budget?

Answer:
Sarah Williams; $750,000.

Sources: company_data.json

Sources:
  - company_data.json
  - company.db table projects
  - company_data.json
  - company.db table projects


In [16]:
result = await ask(rag_pipeline, "What is the budget and status of the RAG Implementation project?")

Question: What is the budget and status of the RAG Implementation project?

Answer:
Budget: $150,000
Status: Active

Sources: [company.db table projects]
Note: The proposal.docx lists a proposed budget of $50,000.

Sources:
  - proposal.docx
  - company.db table projects
  - proposal.docx
  - ai_agents.txt


In [17]:
## A RAG pipeline that only answers from PDF chunks
pdf_rag_pipeline = create_rag_pipeline(vectorstore, filter={"file_type": "pdf"})

result = await ask(pdf_rag_pipeline, "What problem does the Transformer architecture solve?")

Question: What problem does the Transformer architecture solve?

Answer:
The Transformer addresses sequence transduction (e.g., neural machine translation) by replacing recurrence and convolution with self-attention, enabling modeling of long-range dependencies while greatly reducing sequential computation and increasing parallelization.

Sources: 1706.03762v7.pdf p.2–3

Sources:
  - 1706.03762v7.pdf p.3
  - 1706.03762v7.pdf p.2
  - 1706.03762v7.pdf p.2
  - 1706.03762v7.pdf p.2


In [18]:
## Conversational RAG: start a chat with an empty history
conversational_pipeline = create_conversational_rag_pipeline(vectorstore)
chat_history = []

result = await chat(conversational_pipeline, "What is the RAG Implementation project?", chat_history)

Question     : What is the RAG Implementation project?
Searched with: What is the RAG Implementation project?

Answer:
The RAG Implementation project is an initiative to build a retrieval-augmented generation system for internal knowledge search.

Key details:
- Objectives: improve information retrieval accuracy, reduce response time for customer queries, and integrate with the existing knowledge base. [Source: proposal.docx]
- Description: build a retrieval-augmented generation system for internal knowledge search. [Source: company.db table projects]
- Status and priority: Active, High; Department: Engineering. [Source: company.db table projects]
- Budget: $50,000 (proposal) vs. $150,000 (company database). [Sources: proposal.docx; company.db table projects]
- Timeline: proposal estimates 3 months (Research 2 weeks, Development 8 weeks, Testing 2 weeks); database lists 2025-01-15 to 2026-12-31. [Sources: proposal.docx; company.db table projects]
- Team: proposal lists 4 developers + 1

In [19]:
## Follow-up questions: "its" and "it" refer to the RAG Implementation project from the first turn
result = await chat(conversational_pipeline, "What is its budget and status?", chat_history)
result = await chat(conversational_pipeline, "Which technologies does it use?", chat_history)

Question     : What is its budget and status?
Searched with: What is the budget and status of the RAG Implementation project?

Answer:
- Budget: $50,000 (proposal) [Source: proposal.docx]; $150,000 (company record) [Source: company.db table projects]
- Status: Active [Source: company.db table projects]

Sources:
  - proposal.docx
  - company.db table projects
  - proposal.docx
  - ai_agents.txt

Question     : Which technologies does it use?
Searched with: Which technologies does the RAG Implementation project use?

Answer:
- Python (3.8+) [Sources: proposal.docx; company.db table projects]
- LangChain [Source: company.db table projects]
- ChromaDB (vector database) [Sources: proposal.docx; company.db table projects]
- OpenAI API access [Source: proposal.docx]

Sources:
  - proposal.docx
  - company.db table projects
  - proposal.docx
  - ai_agents.txt



In [20]:
## The conversation so far
show_chat_history(chat_history)

# To start a new conversation, empty the history:  chat_history.clear()

6 messages in the history

human: What is the RAG Implementation project?...
   ai: The RAG Implementation project is an initiative to build a retrieval-augmented generation system for...
human: What is its budget and status?...
   ai: - Budget: $50,000 (proposal) [Source: proposal.docx]; $150,000 (company record) [Source: company.db ...
human: Which technologies does it use?...
   ai: - Python (3.8+) [Sources: proposal.docx; company.db table projects] - LangChain [Source: company.db ...


## Reconnecting later

The chunks stay in Astra DB, so another notebook or app can skip the ingestion and go straight to questions:

```python
vectorstore = get_astra_vectorstore(OpenAIEmbeddings(model="text-embedding-3-small"))
rag_pipeline = create_rag_pipeline(vectorstore)

result = await ask(rag_pipeline, "What is multi-head attention?")
```

- **Adding files.** Put new files in `data/raw` and run `run_ingestion(vectorstore)` again. The stable ids mean existing chunks are overwritten, not duplicated.
- **Starting over.** `vectorstore.delete_collection()` deletes the collection and all its data in Astra DB.